In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [13]:
class LlamaMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.hidden_size = config.hidden_size
        self.intermediate_size = config.intermediate_size
        self.gate_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=config.mlp_bias)
        self.up_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=config.mlp_bias)
        self.down_proj = nn.Linear(self.intermediate_size, self.hidden_size, bias=config.mlp_bias)
        # self.act_fn = ACT2FN[config.hidden_act]
        self.act_fn = F.silu

    def forward(self, x):
        down_proj = self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))
        return down_proj

In [29]:
class MixtralSparseMoeBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.hidden_dim = config.hidden_size
        self.ffn_dim = config.intermediate_size
        self.num_experts = config.num_local_experts
        self.top_k = config.num_experts_per_tok

        # gating
        self.gate = nn.Linear(self.hidden_dim, self.num_experts, bias=False)

        # self.experts = nn.ModuleList([MixtralBlockSparseTop2MLP(config) for _ in range(self.num_experts)])
        self.experts = nn.ModuleList([LlamaMLP(config) for _ in range(self.num_experts)])

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        # 将输入 reshape 成二维 [n_tokens, hidden_dim] 然后送入门控网络。输出 router_logits 是各个专家对所有 token 的权重，其形状为 [n_tokens, n_experts] 。
        # import ipdb;ipdb.set_trace()
        batch_size, sequence_length, hidden_dim = hidden_states.shape
        hidden_states = hidden_states.view(-1, hidden_dim) # shape: [n_tokens, hidden_dim]
        router_logits = self.gate(hidden_states) # shape : [n_tokens, n_experts]
        # 用 softmax 将所有专家权重归一化，然后用 top k 得到得分最高的 k 个专家的序号，以及每个专家的权重（按权重排序）。再将每个 token 选中的专家权重归一化。

        # shape: [n_tokens, n_experts]
        routing_weights = F.softmax(router_logits, dim=1, dtype=torch.float)
        # shape: [n_tokens, top_k]
        routing_weights, selected_experts = torch.topk(routing_weights, self.top_k, dim=-1)
        routing_weights /= routing_weights.sum(dim=-1, keepdim=True) # normalization
        routing_weights = routing_weights.to(hidden_states.dtype) # cast back to the input dtype
        # 创建 MOE 的输出，其形状为 [n_tokens, hidden_dim]

        final_hidden_states = torch.zeros(
            (batch_size * sequence_length, hidden_dim), dtype=hidden_states.dtype, device=hidden_states.device
        )
        # 创建 expert mask。首先 one_hot 返回一个形状为 [n_token, top_k, n_experts] 的 tensor，最后一个维度，只有一个元素为 1 表示选中，其余都是 0。假设这个 tensor 叫 T，那么 T[i][j][k] = 1 表示第 i 个 token 选中的所有专家中的第 j 个候选专家（按权重排序）是所有专家中的第 k 个。

        # ; permute 之后，形状为 [n_experts, top_k, n_token]， expert_mask[k][j][i] = 1 表示第 k 个专家，作为第 j 个候选专家，被第 i 个 token 选中。

        # shape: [n_experts, top_k, n_token]
        expert_mask = torch.nn.functional.one_hot(selected_experts, num_classes=self.num_experts).permute(2, 1, 0)
        # ; 5. 找出本次参与计算的专家，因为有些专家没有被任何 token 选中。先对 expert_mask 的最后两个维度求和，如果求和结果大于0，表示这个专家至少被一个 token 选中。用 nonezero() 得到本次参与计算的专家序号。
        
        # shape: [n_hitted, 1]
        expert_hitted = torch.greater(expert_mask.sum(dim=(-1, -2)), 0).nonzero()
        # ; 6. 将 token 送入选中的专家进行计算。遍历所有参与此次计算的专家，得到对应的 token id （top_x）和该 token 的候选专家序号(idx） ，以及 token 对应的 hidden_states（current_state）。将 token 对应的 hidden states 送入选中的专家进行计算，并将计算结果乘以专家权重。最后，将结果按照 token id 累加到 final_hidden_states 里。
        
        # ; 也就是说，每次循环，会用一个专家，计算所有选择该专家的 token，并将结果累加到最终输出上。

        for expert_idx in expert_hitted:
            expert_layer = self.experts[expert_idx]
            idx, top_x = torch.where(expert_mask[expert_idx].squeeze(0))
            current_state = hidden_states[None, top_x].reshape(-1, hidden_dim)
            current_hidden_states = expert_layer(current_state) * routing_weights[top_x, idx, None]
            final_hidden_states.index_add_(0, top_x, current_hidden_states.to(hidden_states.dtype))
        # ; 7. 最后，将输出 reshape 成输入的形状，然后返回 hidden_states 以及归一化前的专家权重。

        final_hidden_states = final_hidden_states.reshape(batch_size, sequence_length, hidden_dim)
        return final_hidden_states, router_logits

In [30]:
class Config:
    def __init__(self):
        self.hidden_size = 512
        self.intermediate_size = 2048
        self.num_local_experts = 8
        self.num_experts_per_tok = 2
        self.mlp_bias = .0

In [31]:
config = Config()
moe = MixtralSparseMoeBlock(config)

In [33]:
final_hidden_states, router_logits = moe(torch.randn([2, 10, 512]))

In [34]:
final_hidden_states.shape, router_logits.shape

(torch.Size([2, 10, 512]), torch.Size([20, 8]))

In [ ]:
class LlamaMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.hidden_size = config.hidden_size
        self.intermediate_size = config.intermediate_size
        self.gate_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=config.mlp_bias)
        self.up_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=config.mlp_bias)
        self.down_proj = nn.Linear(self.intermediate_size, self.hidden_size, bias=config.mlp_bias)
        self.act_fn = ACT2FN[config.hidden_act]

    def forward(self, x):
        down_proj = self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))
        return down_proj